In [1]:
# !pip install -U https://github.com/iqoptionapi/iqoptionapi/archive/refs/heads/master.zip
# !pip install ta

In [2]:
from iqoptionapi.stable_api import IQ_Option
import time
import pandas as pd
import numpy as np
import ta

from enviroment_iq_option import Stock_Market # type: ignore
import math
from datetime import date
import time
from tqdm import trange
import random
from collections import namedtuple, deque

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [3]:
email = "sgtambrosio2019@gmail.com"
password = "YOUR_SECRET_HERE"

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [4]:
# Conectar à API do IQ Option
iqoption = IQ_Option(email, password)
check, reason = iqoption.connect()

/usr/local/lib/python3.10/dist-packages/websocket/_http.py:207: DeprecationWarning: ssl.PROTOCOL_TLS is deprecated
  context = ssl.SSLContext(sslopt.get('ssl_version', ssl.PROTOCOL_SSLv23))


In [5]:
if check:
    print("Conectado com sucesso!")
else:
    print(f"Falha na conexão: {reason}")

Conectado com sucesso!


In [6]:
def get_historical_data(iqoption, ativo, quantidade_total, unidade):
    """
    Obtém dados históricos de velas de um ativo em múltiplas chamadas.

    :param iqoption: Instância da API do IQ Option
    :param ativo: Ativo (ex: "EURUSD")
    :param quantidade_total: Número total de velas a serem obtidas
    :param unidade: Unidade de tempo da vela (ex: 60 para 1 minuto)
    :return: Lista completa de dados das velas
    """
    dados_completos = []
    quantidade_por_chamada = 1000  # Quantidade de velas por chamada (limite da API)
    tempo_atual = int(time.time())  # Timestamp atual

    while quantidade_total > 0:
        quantidade = min(quantidade_por_chamada, quantidade_total)
        candles = iqoption.get_candles(ativo, unidade, quantidade, tempo_atual)
        if not candles:
            break
        dados_completos.extend(candles)
        tempo_atual = candles[-1]['from'] - 1  # Atualiza o timestamp para pegar velas anteriores
        quantidade_total -= quantidade

    return dados_completos

In [7]:
ativo = "EURUSD"
unidade = 60       # segundos
quantidade = 10000 # Quantidade de velas (quantidade de informações)

at : (Timestamp do início do período da vela)    
to : (Timestamp do final do período da vela, 60 segundos depois)             
open : (Preço de abertura da vela)   
close : (Preço de fechamento da vela)    
min : (Preço mínimo durante o período da vela)   
max : (Preço máximo durante o período da vela)    
volume : (Volume de transações durante o período da vela)

In [8]:
# Exemplo: Calcular a Média Móvel Simples (SMA)
def calculate_sma(data, window=10):
    sma = ta.trend.sma_indicator(data['close'], window)
    return sma

# Exemplo: Calcular o RSI
def calculate_rsi(data, window=10):
    rsi = ta.momentum.rsi(data['close'], window)
    return rsi

In [9]:
# dados = get_historical_data(iqoption, ativo, tempo, quantidade, unidade)
dados = get_historical_data(iqoption, ativo, quantidade, unidade)

# Converter para DataFrame
data = pd.DataFrame(dados)

# Converter timestamps para datetime
data['from'] = pd.to_datetime(data['from'], unit='s')

data.drop(["from", "id", "at", "to", "volume"], axis=1, inplace=True)

In [10]:
sma = calculate_sma(data)
rsi = calculate_rsi(data)

data['SMA'] = sma
data['RSI'] = rsi
data.dropna(inplace=True)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [11]:
data.head()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,open,close,min,max,SMA,RSI
9,1.069830,1.069765,1.069725,1.069845,1.069796,46.015393
10,1.069770,1.069725,1.069705,1.069835,1.069791,40.389548
11,1.069735,1.069725,1.069705,1.069795,1.069783,40.389548
12,1.069715,1.069695,1.069645,1.069755,1.069780,36.282260
13,1.069705,1.069690,1.069655,1.069745,1.069768,35.611628


In [12]:
data = np.array(data.astype(float), dtype="float32")

env = Stock_Market(data)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [13]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [14]:
class ReplayBuffer(object):
    """Classe para armazenar e amostrar transições de memória de repetição"""

    def __init__(self, capacity):
        # Inicializa uma deque (fila de duas extremidades) com um tamanho máximo de `capacity`
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Salva uma transição"""
        # Adiciona uma nova transição à memória
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        """Amostra um lote de transições aleatoriamente"""
        # Retorna um número especificado (batch_size) de transições aleatórias da memória
        return random.sample(self.memory, batch_size)

    def __len__(self):
        """Retorna o tamanho atual da memória"""
        # Retorna o número de transições atualmente armazenadas na memória
        return len(self.memory)

In [15]:
BUFFER_SIZE = int(len(env))  # Tamanho do buffer de replay
memory = ReplayBuffer(BUFFER_SIZE) # Inicializa a memória de replay com a capacidade especificada

In [16]:
class DQN(nn.Module):
    """Rede Neural Profunda para a DQN (Deep Q-Network)"""

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        # Primeira camada totalmente conectada (input para hidden)
        self.layer1 = nn.Linear(n_observations, 128)
        # Segunda camada totalmente conectada (hidden para hidden)
        self.layer2 = nn.Linear(128, 128)
        # Terceira camada totalmente conectada (hidden para output)
        self.layer3 = nn.Linear(128, n_actions)

    # Chamado com um elemento para determinar a próxima ação, ou um lote durante a otimização
    # Retorna tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        # Aplicar a função de ativação ReLU após a primeira camada
        x = F.relu(self.layer1(x))
        # Aplicar a função de ativação ReLU após a segunda camada
        x = F.relu(self.layer2(x))
        # Passar pela terceira camada sem ativação para obter os valores Q para cada ação
        return self.layer3(x)

In [17]:
n_actions = env.action_space
state = env.reset()
n_observations = len(state)

In [18]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [19]:
# Cria a rede neural de política e move para o dispositivo especificado (CPU ou GPU)
policy_net = DQN(n_observations, n_actions).to(device)

# Cria a rede neural alvo e move para o dispositivo especificado
target_net = DQN(n_observations, n_actions).to(device)

# Inicializa a rede neural alvo com os mesmos pesos da rede de política
target_net.load_state_dict(policy_net.state_dict())

<All keys matched successfully>

In [20]:
LR = 1e-4  # LR é a taxa de aprendizado do otimizador ``AdamW``
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)

In [21]:
BATCH_SIZE = 128 # BATCH_SIZE é o número de transições amostradas do buffer de reprodução
GAMMA = 0.99    # GAMMA é o fator de desconto
EPS_START = 0.9  # EPS_START é o valor inicial do épsilon
EPS_END = 0.01   # EPS_END é o valor final do épsilon
EPS_DECAY = 1000 # EPS_DECAY controla a taxa de decaimento exponencial do épsilon, maior significa um decaimento mais lento
TAU = 0.005      # TAU é a taxa de atualização da rede alvo

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [22]:
steps_done = 0  # Inicializa o contador de passos

def select_action(state):
    global steps_done  # Usa a variável global steps_done para manter o controle do número de passos realizados
    sample = random.random()  # Gera um número aleatório entre 0 e 1
    # Calcula o valor de epsilon (probabilidade de escolher uma ação aleatória)
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1  # Incrementa o contador de passos

    # Verifica se o número aleatório é maior que o epsilon
    if sample > eps_threshold:
        # Se o número aleatório for maior que o epsilon,
        # selecione a ação com maior recompensa esperada (ação "greedy")
        with torch.no_grad():
            # policy_net(state) retorna um tensor de valores Q para cada ação
            # t.max(1) retorna o valor máximo em cada linha, max(1) retorna (valores, indices)
            # .indices.view(1, 1) retorna o índice da ação com maior valor Q
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        # Caso contrário, selecione uma ação aleatória
        return torch.tensor([[env.sample()]], device=device, dtype=torch.long)

In [23]:
def optimize_model():
    # Verifica se há transições suficientes no replay memory para formar um batch
    if len(memory) < BATCH_SIZE:
        return  # Se não houver, retorna sem fazer nada

    # Amostra um batch de transições do replay memory
    transitions = memory.sample(BATCH_SIZE)

    # Transpõe o batch para obter um Transition de batch-arrays em vez de um batch-array de Transitions
    # Lembre-se que Transition tem forma: Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))
    # Isso facilita o trabalho com os dados em PyTorch
    batch = Transition(*zip(*transitions))

    # Cria uma máscara de estados não finais e concatena os elementos do batch
    # Um estado final é aquele em que a simulação terminou
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                            batch.next_state)), device=device, dtype=torch.bool)

    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Computa Q(s_t, a) - o modelo computa Q(s_t), então selecionamos as colunas das ações tomadas
    # Estas são as ações que teriam sido tomadas para cada estado do batch de acordo com policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Computa os retornos para todos os próximos estados.
    # Valores esperados das ações para non_final_next_states são computados com  base no target_net "mais antigo"
    # Selecionando a melhor recompensa com max(1).values
    # Isso é mesclado com base na máscara, de modo que teremos o valor do estado esperado ou 0 caso o estado fosse final
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values

    # Computa os valores Q esperados
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Computa a perda Huber
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    ################# Otimiza o modelo #################
    optimizer.zero_grad()  # Zera os gradientes acumulados
    loss.backward()        # Computa os gradientes
    # Clipping dos gradientes para evitar explosão de gradientes
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()       # Atualiza os pesos do modelo

In [ ]:
%%time

# Definindo variáveis

num_episodes = 100 # Número de episódios
num_states = len(env)
rewards = []

# Loop principal de episódios
for episode in trange(num_episodes):
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    profit_episode = 0

    for t in range(num_states):
        action = select_action(state)
        observation, reward, terminated = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        done = terminated

        next_state = None if done else torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
        memory.push(state, action, next_state, reward)  # Armazena a transição na memória
        state = next_state  # Move para o próximo estado

        if t % 10 == 0:
          optimize_model()  # Executa uma etapa de otimização

        profit_episode += reward.item()

        # Atualização suave dos pesos da rede alvo
        for target_param, policy_param in zip(target_net.parameters(), policy_net.parameters()):
            target_param.data.copy_(policy_param.data * TAU + target_param.data * (1 - TAU))

        if done:
          break

    if (episode + 1) == num_episodes:
      torch.save(target_net.state_dict(), 'weights_ia_trader.pth')
      torch.save(target_net, 'agent_iq_option.pth')

    rewards.append(profit_episode)
    print(f"\nLucro no episódio {episode + 1}: R$ {profit_episode:.2f}")
    print(f"Lucro Acumulado: R$ {sum(rewards):.2f}")

total = sum(rewards)
print(f'Retorno total: R$ {total :.2f}')